In [0]:
# =====================================================
# IMPORTAÇÃO DAS BIBLIOTECAS
# =====================================================

# Funções SQL do Spark
from pyspark.sql import functions as F

# Estrutura de linhas para criação do DataFrame final
from pyspark.sql import Row

# Data e hora da execução
from datetime import datetime

In [0]:
# =====================================================
# LEITURA DAS TABELAS DA CAMADA BRONZE
# =====================================================

customers = spark.table("projeto.bronze.customers")
orders = spark.table("projeto.bronze.orders")
order_items = spark.table("projeto.bronze.order_items")
payments = spark.table("projeto.bronze.order_payments")
reviews = spark.table("projeto.bronze.order_reviews")
products = spark.table("projeto.bronze.products")

In [0]:
# =====================================================
# CONFIGURAÇÃO DAS TABELAS
# =====================================================

bases = {

    "customers": {
        "df": customers,
        "pk": ["customer_id"]
    },

    "orders": {
        "df": orders,
        "pk": ["order_id"]
    },

    "order_items": {
        "df": order_items,
        "pk": ["order_id", "order_item_id"]
    },

    "payments": {
        "df": payments,
        "pk": ["order_id"]
    },

    "reviews": {
        "df": reviews,
        "pk": ["review_id"]
    },

    "products": {
        "df": products,
        "pk": ["product_id"]
    }

}

In [0]:
# =====================================================
# FUNÇÃO DE DUPLICIDADE
# =====================================================

def calcular_duplicidade(df, chave):

    """
    Recebe:
        DataFrame
        Chave primária

    Retorna:
        Quantidade de registros duplicados
    """

    duplicados = (
        df.groupBy(chave)
          .count()
          .filter(F.col("count") > 1)
          .count()
    )

    return duplicados

In [0]:
# =====================================================
# FUNÇÃO PARA CÁLCULO DE NULOS
# =====================================================

def calcular_nulos(df, coluna):

    """
    Calcula a quantidade de registros nulos
    para uma coluna específica.
    """

    return df.filter(
        F.col(coluna).isNull()
    ).count()

In [0]:
# =====================================================
# PROCESSAMENTO DE DATA QUALITY
# =====================================================

resultado = []

# Data de execução da análise
data_execucao = datetime.now()

# Percorre todas as tabelas configuradas
for nome_tabela, config in bases.items():

    print(f"Processando {nome_tabela}")

    # DataFrame atual
    df = config["df"]

    # Chave primária
    chave = config["pk"]

    # Quantidade total de registros
    total_registros = df.count()

    # Duplicidades da tabela
    qtd_duplicados = calcular_duplicidade(df, chave)

    # Percentual de duplicidade
    pct_duplicados = round(
        (qtd_duplicados / total_registros) * 100,
        2
    )

    # Percorre todas as colunas
    for coluna in df.columns:

        # Quantidade de nulos
        qtd_nulos = calcular_nulos(df, coluna)

        # Percentual de nulos
        pct_nulos = round(
            (qtd_nulos / total_registros) * 100,
            2
        )

        resultado.append(
            Row(
                data_execucao=data_execucao,
                tabela=nome_tabela,
                coluna=coluna,
                total_registros=total_registros,
                duplicados=qtd_duplicados,
                pct_duplicados=pct_duplicados,
                qtd_nulos=qtd_nulos,
                pct_nulos=pct_nulos
            )
        )

Processando customers
Processando orders
Processando order_items
Processando payments
Processando reviews
Processando products


In [0]:
# =====================================================
# GERAÇÃO DO DATAFRAME FINAL
# =====================================================

dq_report = spark.createDataFrame(resultado)

display(dq_report)

data_execucao,tabela,coluna,total_registros,duplicados,pct_duplicados,qtd_nulos,pct_nulos
2026-10-30T13:25:20.574Z,customers,customer_id,99441,0,0.0,0,0.0
2026-10-30T13:25:20.574Z,customers,customer_unique_id,99441,0,0.0,0,0.0
2026-10-30T13:25:20.574Z,customers,customer_zip_code_prefix,99441,0,0.0,0,0.0
2026-10-30T13:25:20.574Z,customers,customer_city,99441,0,0.0,0,0.0
2026-10-30T13:25:20.574Z,customers,customer_state,99441,0,0.0,0,0.0
2026-10-30T13:25:20.574Z,orders,order_id,99441,0,0.0,0,0.0
2026-10-30T13:25:20.574Z,orders,customer_id,99441,0,0.0,0,0.0
2026-10-30T13:25:20.574Z,orders,order_status,99441,0,0.0,0,0.0
2026-10-30T13:25:20.574Z,orders,order_purchase_timestamp,99441,0,0.0,0,0.0
2026-10-30T13:25:20.574Z,orders,order_approved_at,99441,0,0.0,160,0.16


In [0]:
# =====================================================
# SALVANDO RESULTADO
# =====================================================

spark.sql("CREATE SCHEMA IF NOT EXISTS projeto.silver")

dq_report.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("projeto.silver.data_quality_report")

In [0]:
#1. Leitura das Tabelas Silver
# =====================================================
# TECH CHALLENGE - FASE 4
# VOICE OF CUSTOMER INTELLIGENCE
# CAMADA GOLD - BASE DE CONHECIMENTO
# =====================================================

from pyspark.sql import functions as F

# =====================================================
# LEITURA DAS TABELAS
# =====================================================

customers = spark.table("bronze.customers")
orders = spark.table("bronze.orders")
order_items = spark.table("bronze.order_items")
products = spark.table("bronze.products")
reviews = spark.table("bronze.order_reviews")

2. Seleção dos Campos Relevantes

Não precisamos carregar tudo.

O PDF é sobre Voice of Customer.

Portanto vamos focar nas variáveis relevantes.

In [0]:
#
# =====================================================
# PEDIDOS
# =====================================================

orders_silver = orders.select(
    "order_id",
    "customer_id",
    "order_status",
    "order_purchase_timestamp",
    "order_delivered_customer_date"
)

# =====================================================
# REVIEWS
# =====================================================

reviews_silver = reviews.select(
    "review_id",
    "order_id",
    "review_score",
    "review_comment_title",
    "review_comment_message"
)

# =====================================================
# PRODUTOS
# =====================================================

products_silver = products.select(
    "product_id",
    "product_category_name"
)

# =====================================================
# ITENS DO PEDIDO
# =====================================================

items_silver = order_items.select(
    "order_id",
    "product_id"
)

In [0]:
3. Tratamento das Avaliações

RAG não gosta de dados sujos.

In [0]:
reviews_silver = reviews_silver.filter(
    F.col("review_comment_message").isNotNull()
)

In [0]:
reviews_silver = reviews_silver.withColumn(
    "review_comment_message",
    F.trim(F.col("review_comment_message"))
)

In [0]:
Remover registros vazios

In [0]:
#verificação de integridade# =====================================================
# PEDIDOS SEM CLIENTE
# =====================================================

orders_sem_customer = (
    orders.join(
        customers,
        on="customer_id",
        how="left_anti"
    )
)

qtd_orders_sem_customer = orders_sem_customer.count()

4. Montagem da Base Analítica
Orders + Reviews

In [0]:
base_reviews = (
    reviews_silver
    .join(
        orders_silver,
        on="order_id",
        how="left"
    )
)

Adicionando Produto

In [0]:
base_reviews = (
    base_reviews
    .join(
        items_silver,
        on="order_id",
        how="left"
    )
    .join(
        products_silver,
        on="product_id",
        how="left"
    )
)

5. Criar Indicador de Sentimento

Muito útil para a banca.

Python

In [0]:
# try_cast tolera registros com review_score deslocado (texto do CSV mal parseado)
base_reviews = (
    base_reviews
    .withColumn(
        "sentimento",
        F.when(F.expr("try_cast(review_score as int)") >= 4, "POSITIVO")
         .when(F.expr("try_cast(review_score as int)") == 3, "NEUTRO")
         .otherwise("NEGATIVO")
    )
)

6. Calcular Atraso da Entrega

Outro metadado valioso.

Python

In [0]:
base_reviews = (
    base_reviews
    .withColumn(
        "dias_entrega",
        F.datediff(
            F.col("order_delivered_customer_date"),
            F.col("order_purchase_timestamp")
        )
    )
)

7. Criar Documento para o RAG

Aqui está a lógica principal.

Transformamos várias colunas em um documento.

In [0]:
base_reviews = (
    base_reviews
    .withColumn(
        "document_text",

        F.concat_ws(
            " ",

            F.lit("Categoria do Produto:"),

            F.coalesce(
                F.col("product_category_name"),
                F.lit("Não Informado")
            ),

            F.lit("Nota da Avaliação:"),

            F.col("review_score"),

            F.lit("Sentimento:"),

            F.col("sentimento"),

            F.lit("Comentário do Cliente:"),

            F.col("review_comment_message")
        )
    )
)

8. Criar Identificador do Documento
Python

In [0]:
base_reviews = (
    base_reviews
    .withColumn(
        "document_id",
        F.monotonically_increasing_id()
    )
)

9. Estrutura Final da Knowledge Base
Python

In [0]:
knowledge_base = base_reviews.select(

    "document_id",
    "review_id",
    "order_id",
    "review_score",
    "sentimento",
    "product_category_name",
    "dias_entrega",
    "document_text"

)

10. Persistir Gold

In [0]:
spark.sql("USE CATALOG projeto")

spark.sql("CREATE SCHEMA IF NOT EXISTS projeto.gold")

knowledge_base.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "gold.customer_reviews_knowledge_base"
    )

11. Estatísticas para a Banca

In [0]:
# review_score chega como STRING e 106 registros têm texto deslocado (CSV mal parseado);
# try_cast devolve NULL nesses casos em vez de interromper a contagem
kb = (
    knowledge_base
    .withColumn("score_num", F.expr("try_cast(review_score as int)"))
    .filter(F.col("score_num").isNotNull())
    .withColumn(
        "sentimento",
        F.when(F.col("score_num") >= 4, "POSITIVO")
         .when(F.col("score_num") == 3, "NEUTRO")
         .otherwise("NEGATIVO")
    )
)

print("Quantidade de documentos:")
print(knowledge_base.count())

print("Avaliações Positivas:")
print(
    kb.filter(
        F.col("sentimento") == "POSITIVO"
    ).count()
)

print("Avaliações Negativas:")
print(
    kb.filter(
        F.col("sentimento") == "NEGATIVO"
    ).count()
)

Quantidade de documentos:
48272
Avaliações Positivas:
29514
Avaliações Negativas:
14322


Leitura ++ OPTI

In [0]:
from pyspark.sql import functions as F

kb = spark.table(
    "gold.customer_reviews_knowledge_base"
)

display(kb.limit(5))

document_id,review_id,order_id,review_score,sentimento,product_category_name,dias_entrega,document_text
0,72730619a00e0ecdecf0a4ff862d8996,af76a45a10524549102e010f2bd9d3cd,1,NEGATIVO,papelaria,14,Categoria do Produto: papelaria Nota da Avaliação: 1 Sentimento: NEGATIVO Comentário do Cliente: comprei tres pacotes de cinco folhas cada de papel transfer para tecido escuro e so recebi dois
1,346317f9aff540395fe46893611854f8,546ffb713086001ce9f0009158f88b73,1,NEGATIVO,esporte_lazer,3,Categoria do Produto: esporte_lazer Nota da Avaliação: 1 Sentimento: NEGATIVO Comentário do Cliente: Não funciona não faz sincronismo
2,7914fa319f12631679d1704d7dcd17df,06ecfa8dfae87263e05d953aa561603d,5,POSITIVO,automotivo,6,Categoria do Produto: automotivo Nota da Avaliação: 5 Sentimento: POSITIVO Comentário do Cliente: Recomendo
3,49f6f0bbf53f6d961deefc4d2fb0a6cd,4b9d1bbbeed0481ecc124c66e355896f,5,POSITIVO,moveis_quarto,8,Categoria do Produto: moveis_quarto Nota da Avaliação: 5 Sentimento: POSITIVO Comentário do Cliente: Chegou antes do prazo é a cabeceira é linda.
4,1545c0d2c381662ba869e91e4c1aaca1,c589bcc1414f7d7d949f353b1d1a45d9,5,POSITIVO,esporte_lazer,56,Categoria do Produto: esporte_lazer Nota da Avaliação: 5 Sentimento: POSITIVO Comentário do Cliente: Até agora tudo bem estou esperando a entrega do produto


In [0]:
%pip install sentence-transformers faiss-cpu
dbutils.library.restartPython()

Looking in indexes: [REDACTED]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 760.7/760.7 kB 18.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.0/857.0 kB 34.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.1/4.1 MB 118.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 199.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 142.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 173.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 805.9/805.9 kB 39.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 483.7/483.7 MB 134.9 MB/s  0:00:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 595.3/595.3 MB 90.0 MB/s  0:00:04
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 221.1/221.1 MB 97.7 MB/s  0:00:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.0/216.0 MB 92.3 MB/s  0:00:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.2/135.2 MB 101.3 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Carregar Modelo

In [0]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "paraphrase-multilingual-MiniLM-L12-v2"
)

/local_disk0/.ephemeral_nfs/envs/pythonEnv-2a8f800b-a9d6-4315-8a4a-ff95c6f885d3/lib/python3.12/site-packages/torch/_vmap_internals.py:9: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  from torch.utils._pytree import _broadcast_to_and_flatten, tree_flatten, tree_unflatten


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

... (4 outputs left out)

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [0]:
kb_documents = kb.select(
    "document_id",
    "document_text"
)

Aproveita melhor CPU/GPU do cluster.

In [0]:
pandas_df = kb_documents.toPandas()

embeddings = embedding_model.encode(
    pandas_df["document_text"].tolist(),
    batch_size = 128,
    show_progress_bar = True
)

Batches:   0%|          | 0/378 [00:00<?, ?it/s]

In [0]:
pandas_df["embedding"] = embeddings.tolist()

In [0]:
embedding_df = spark.createDataFrame(
    pandas_df
)

In [0]:
embedding_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "gold.customer_reviews_embeddings"
    )

Criação do FAISS

In [0]:
embeddings_df = spark.table(
    "gold.customer_reviews_embeddings"
)

Converter somente o necessário

In [0]:
pdf = embeddings_df.select(
    "document_id",
    "document_text",
    "embedding"
).toPandas()

Criar Matriz Numérica

In [0]:
import numpy as np

vectors = np.array(
    pdf["embedding"].tolist()
).astype("float32")

Criar Índice

In [0]:
import faiss

dimension = vectors.shape[1]

index = faiss.IndexFlatL2(
    dimension
)

Adicionar Vetores

In [0]:
index.add(vectors)

print(
    f"Vetores indexados: {index.ntotal}"
)

Vetores indexados: 48272


In [0]:
import os
os.makedirs('/Volumes/projeto/tech_challenger_4/projeto_4_/rag/', exist_ok=True)
faiss.write_index(
    index,
    "/Volumes/projeto/tech_challenger_4/projeto_4_/rag/faiss_olist.index"
)

Salvar Metadados

In [0]:
meta = pdf[
    [
        "document_id",
        "document_text"
    ]
]

# toPandas() deixa métricas do plano Spark em .attrs, que não são serializáveis no parquet
meta.attrs = {}

meta.to_parquet(
    "/Volumes/projeto/tech_challenger_4/projeto_4_/rag/metadata.parquet",
    index=False
)

Carregar tudo novamente

In [0]:
import faiss
import pandas as pd

index = faiss.read_index(
    "/Volumes/projeto/tech_challenger_4/projeto_4_/rag/faiss_olist.index"
)

metadata = pd.read_parquet(
    "/Volumes/projeto/tech_challenger_4/projeto_4_/rag/metadata.parquet"
)

Modelo de embeddings
Python

In [0]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "paraphrase-multilingual-MiniLM-L12-v2"
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Função de busca

In [0]:
def buscar_documentos(
        pergunta,
        top_k=5
):
    
    pergunta_embedding = embedding_model.encode(
        [pergunta]
    ).astype("float32")
    
    distancias, indices = index.search(
        pergunta_embedding,
        top_k
    )
    
    resultado = metadata.iloc[
        indices[0]
    ]
    
    return resultado


TESTE

In [0]:
buscar_documentos(
    "problemas de atraso na entrega"
)


,document_id,document_text
13621,19655,Categoria do Produto: utilidades_domesticas No...
23011,4909,Categoria do Produto: eletrodomesticos Nota da...
2431,32601,Categoria do Produto: moveis_decoracao Nota da...
35552,29518,Categoria do Produto: moveis_decoracao Nota da...
10643,10643,Categoria do Produto: cama_mesa_banho Nota da ...


In [0]:
# =====================================================
# ITENS SEM PEDIDO
# =====================================================

items_sem_order = (
    order_items.join(
        orders,
        on="order_id",
        how="left_anti"
    )
)

qtd_items_sem_order = items_sem_order.count()

In [0]:
# =====================================================
# ITENS SEM PRODUTO
# =====================================================

items_sem_produto = (
    order_items.join(
        products,
        on="product_id",
        how="left_anti"
    )
)

qtd_items_sem_produto = items_sem_produto.count()

In [0]:
#/Volumes/projeto/tech_challenger_4/projeto_4_/
base_path = "/Volumes/projeto/tech_challenger_4/projeto_4_/"

arquivos = {
    "customers": "olist_customers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "order_payments": "olist_order_payments_dataset.csv",
    "order_reviews": "olist_order_reviews_dataset.csv",
    "orders": "olist_orders_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "category_translation": "product_category_name_translation.csv"
}

spark.sql("CREATE SCHEMA IF NOT EXISTS projeto.bronze")

for tabela, arquivo in arquivos.items():

    df = (
        spark.read
        .option("header","true")
        .option("inferSchema","true")
        .csv(base_path + arquivo)
    )

    df.write \
      .format("delta") \
      .mode("overwrite") \
      .saveAsTable(f"projeto.bronze.{tabela}")